# 🎁 Decorator Design Pattern — Hinglish Beginner Guide

**Decorator Design Pattern** ek **structural design pattern** hai jo kisi object mein 
**runtime pe naya behavior/feature add** karne deta hai — **without modifying** uski original class.

> **Simple bhasha mein:** Socho tum ek **plain coffee** order karte ho. Ab agar tumhe **Milk** aur 
> **Sugar** bhi chahiye, to tum coffee ki original class ko change nahi karte — balki usko 
> **"wrap"** kar dete ho ek Milk layer se, phir uske upar ek Sugar layer se. Har layer apna 
> kaam (extra cost + description) add karti hai, bina andar wali coffee ko touch kiye.

Is notebook mein hum seekhenge:
1. Decorator Pattern kya hai aur kyun use karte hain
2. **Bad/Without Pattern approach** — problem kya hai wo dekhenge
3. Text-based UML diagrams (comments ke andar)
4. **Best/With Pattern approach** — step-by-step Python implementation
5. Advantages/Disadvantages
6. **5 Practice Questions with Answers** (real-world examples ke saath)


## ☕ Real-Life Analogy (Samjhne ke liye)

Coffee Shop mein order karte time:

- Base: **Plain Coffee** — ₹2.0
- + **Milk** add karo — ₹0.5 extra
- + **Sugar** add karo — ₹0.2 extra
- + **Whipped Cream** add karo — ₹0.7 extra

Customer kisi bhi combination mein add-ons choose kar sakta hai. Agar hum har combination 
ke liye **alag class** banaye (jaise `CoffeeWithMilk`, `CoffeeWithMilkAndSugar`, 
`CoffeeWithMilkSugarCream`...), to combinations badhte hi **classes ka explosion** ho jayega! 💥

**Decorator Pattern isko solve karta hai** — har add-on ek chhota "wrapper" (decorator) hota hai 
jo original coffee object ko **wrap** karta hai aur apna extra cost/description add kar deta hai. 
Jitna chaho utna **stack/combine** kar sakte ho, **runtime pe**, bina koi nayi class banaye!

**Doosre real-world examples:**
- 🎥 Video Streaming: Video pe Subtitles, Audio Enhancement wrap karna
- ✍️ Text Editor: Text pe Bold, Italic, Underline wrap karna
- 📂 Java I/O Streams: `FileInputStream` ko `BufferedInputStream` se wrap karna


## ❌ Step 1: Problem — BINA Decorator Pattern ke (Bad Approach)

Chalo pehle dekhte hain agar hum **naive tarike** se (inheritance/subclassing se) 
ye problem solve karne ki koshish karein, to kya dikkat aati hai.


In [ ]:
"""
BAD APPROACH — Inheritance se har combination ke liye alag class banana
=============================================================================
Idea: Har add-on combination ke liye ek NAYI SUBCLASS banate hain.
"""

class PlainCoffee:
    def get_description(self):
        return "Plain Coffee"
    def get_cost(self):
        return 2.0

# Ab Milk add karna hai -> ek nayi subclass banao
class CoffeeWithMilk(PlainCoffee):
    def get_description(self):
        return super().get_description() + ", Milk"
    def get_cost(self):
        return super().get_cost() + 0.5

# Ab Sugar bhi add karna hai -> Milk wali class ko phirse extend karo
class CoffeeWithMilkAndSugar(CoffeeWithMilk):
    def get_description(self):
        return super().get_description() + ", Sugar"
    def get_cost(self):
        return super().get_cost() + 0.2

# Ab Whipped Cream bhi chahiye -> phir se nayi class!
class CoffeeWithMilkSugarAndCream(CoffeeWithMilkAndSugar):
    def get_description(self):
        return super().get_description() + ", Whipped Cream"
    def get_cost(self):
        return super().get_cost() + 0.7


# Test
c = CoffeeWithMilkAndSugar()
print(c.get_description(), "-> $", c.get_cost())

"""
❌ PROBLEMS WITH THIS BAD APPROACH:

1. CLASS EXPLOSION: Agar 4 add-ons hain (Milk, Sugar, Cream, Chocolate), to
   possible combinations = 2^4 = 16! Har combination ke liye alag class banani
   padegi -> CoffeeWithMilk, CoffeeWithSugar, CoffeeWithMilkSugar,
   CoffeeWithMilkCream, CoffeeWithSugarCream, ... (bahut zyada classes!)

2. NO RUNTIME FLEXIBILITY: Customer agar order karte waqt bole
   "Sugar hata do, Cream add kar do" -> humare paas koi
   'CoffeeWithMilkAndCream' (without sugar) class nahi hai, to naya banana padega.

3. TIGHT COUPLING (Fixed Order): CoffeeWithMilkAndSugar hamesha
   pehle Milk phir Sugar add karega -- order change nahi kar sakte
   (jaise "pehle Sugar, phir Milk" wala order impossible hai is design mein).

4. VIOLATES OPEN/CLOSED PRINCIPLE: Naya add-on (jaise Chocolate) add karne ke
   liye EXISTING classes ko dobara likhna/extend karna padta hai in complex ways.

LESSON: Static inheritance "combinations" ke liye achha nahi hai.
Humein kuch chahiye jo RUNTIME pe, FLEXIBLE order mein, features combine kar sake.
Yahi Decorator Pattern solve karta hai! Chalo best approach dekhte hain 👇
"""


In [ ]:
"""
TEXT-BASED UML CLASS DIAGRAM — Decorator Design Pattern (BEST APPROACH)
=================================================================================

    +---------------------------+
    |         Coffee                |
    |     (Component - ABC)          |
    +---------------------------+
    | + get_description() (abstract)  |
    | + get_cost()        (abstract)   |
    +-------------^---------------+
                  |
      +-----------+------------------------------+
      |                                           |
+-----+--------+                     +------------+-------------+
|  PlainCoffee   |                     |     CoffeeDecorator          |
| (Concrete       |                     |  (Abstract Decorator, ABC)     |
|  Component)     |                     +------------------------------+
+----------------+                     | - decorated_coffee : Coffee      |
| + get_description()|                  | + get_description() (delegate)    |
|   -> "Plain Coffee"  |                 | + get_cost()        (delegate)     |
| + get_cost() -> 2.0    |               +---------------^--------------+
+----------------+                                      |
                                        +-----------------+------------------+
                                        |                                    |
                              +---------+---------+              +----------+---------+
                              |   MilkDecorator      |              |   SugarDecorator     |
                              | (Concrete Decorator)   |              | (Concrete Decorator)  |
                              +-----------------------+              +-----------------------+
                              | + get_description()       |          | + get_description()        |
                              |   -> ..., "Milk"             |         |   -> ..., "Sugar"            |
                              | + get_cost() -> +0.5            |     | + get_cost() -> +0.2           |
                              +-----------------------+          +-----------------------+


HOW WRAPPING WORKS (Object Composition, layer by layer)
================================================================

  sugar_milk_coffee = SugarDecorator( MilkDecorator( PlainCoffee() ) )

    +-----------------------------------------------------------+
    |  SugarDecorator                                              |
    |   +-------------------------------------------------------+  |
    |   |  MilkDecorator                                            |  |
    |   |    +-------------------------------------------------+ |  |
    |   |    |  PlainCoffee                                       | |  |
    |   |    |  (description="Plain Coffee", cost=2.0)              | |  |
    |   |    +-------------------------------------------------+ |  |
    |   |   adds ", Milk" and +0.5                                    |  |
    |   +-------------------------------------------------------+  |
    |  adds ", Sugar" and +0.2                                        |
    +-----------------------------------------------------------+

  get_cost() call chain (andar se bahar tak calculate hota hai):
    SugarDecorator.get_cost()
        -> calls MilkDecorator.get_cost()
              -> calls PlainCoffee.get_cost() -> returns 2.0
           -> 2.0 + 0.5 = 2.5
        -> 2.5 + 0.2 = 2.7   (FINAL ANSWER)
"""

print("UML aur wrapping diagram upar comment mein hai — dhyan se padho! 👆")


## 🧩 4 Main Components (Yaad rakhne wali cheez)

| Component | Kaam kya hai | Analogy (Coffee example) |
|---|---|---|
| **Component (Interface)** | Common operations define karta hai jo sab implement karenge | `Coffee` (abstract) |
| **Concrete Component** | Core/base object jisme basic functionality hoti hai | `PlainCoffee` |
| **Decorator (Abstract)** | Wrapper class jo Component ko hold karti hai aur behavior add karti hai | `CoffeeDecorator` |
| **Concrete Decorator** | Specific extra feature add karta hai | `MilkDecorator`, `SugarDecorator` |

> **Key Insight:** Decorator khud bhi Component interface implement karta hai — 
> isliye tum Decorator ke upar **aur ek Decorator** wrap kar sakte ho! Yahi wajah hai 
> ki tum jitne chaho utne layers **stack** kar sakte ho (Milk pe Sugar pe Cream...).


## ✅ Step 2: Solution — WITH Decorator Pattern (Best Approach)

Ab dekhte hain **best approach** — Decorator Pattern use karke isi problem ko 
**clean, flexible aur runtime-combinable** tareeke se solve karna.

### Implementation Steps:
1. **Component Interface** banao (Coffee)
2. **Concrete Component** banao (PlainCoffee)
3. **Abstract Decorator** banao (CoffeeDecorator) — jo Component ko wrap kare
4. **Concrete Decorators** banao (MilkDecorator, SugarDecorator) — jo actual feature add karein
5. **Client** — jitne chaho utne decorators **stack/combine** karo, runtime pe!


In [ ]:
"""
STEP 1: COMPONENT INTERFACE
=================================
Ye common contract hai jo Concrete Component AUR Decorator dono follow karenge.
"""

from abc import ABC, abstractmethod

class Coffee(ABC):
    @abstractmethod
    def get_description(self):
        pass

    @abstractmethod
    def get_cost(self):
        pass


In [ ]:
"""
STEP 2: CONCRETE COMPONENT
================================
Ye base/core object hai — sabse andar wali layer.
"""

class PlainCoffee(Coffee):
    def get_description(self):
        return "Plain Coffee"

    def get_cost(self):
        return 2.0


In [ ]:
"""
STEP 3: ABSTRACT DECORATOR
================================
Ye bhi Coffee interface implement karta hai (isliye Decorator khud bhi
"ek Coffee" ki tarah use ho sakta hai -> isse STACKING possible hoti hai).

Ye ek "decorated_coffee" reference store karta hai (wrapped object),
aur by default sirf usi ko DELEGATE (forward) kar deta hai.
"""

class CoffeeDecorator(Coffee):
    def __init__(self, decorated_coffee):
        self.decorated_coffee = decorated_coffee   # <-- wrapped object

    def get_description(self):
        return self.decorated_coffee.get_description()   # delegate

    def get_cost(self):
        return self.decorated_coffee.get_cost()           # delegate


In [ ]:
"""
STEP 4: CONCRETE DECORATORS
=================================
Ye actual "add-ons" hain. Har ek apna extra description/cost add karta hai,
UPAR se decorated_coffee ka result use karke (super() jaisa hi delegate pattern).
"""

class MilkDecorator(CoffeeDecorator):
    def get_description(self):
        return self.decorated_coffee.get_description() + ", Milk"

    def get_cost(self):
        return self.decorated_coffee.get_cost() + 0.5


class SugarDecorator(CoffeeDecorator):
    def get_description(self):
        return self.decorated_coffee.get_description() + ", Sugar"

    def get_cost(self):
        return self.decorated_coffee.get_cost() + 0.2


class WhippedCreamDecorator(CoffeeDecorator):
    def get_description(self):
        return self.decorated_coffee.get_description() + ", Whipped Cream"

    def get_cost(self):
        return self.decorated_coffee.get_cost() + 0.7


In [ ]:
"""
STEP 5: CLIENT CODE
=========================
Ab dekho ye kितna FLEXIBLE hai — koi nayi class banaye bina,
RUNTIME pe kisi bhi combination mein add-ons stack kar sakte hain!
"""

def main():
    # Sirf Plain Coffee
    coffee = PlainCoffee()
    print(f"Description: {coffee.get_description()}")
    print(f"Cost: ${coffee.get_cost()}\n")

    # Coffee + Milk
    milk_coffee = MilkDecorator(PlainCoffee())
    print(f"Description: {milk_coffee.get_description()}")
    print(f"Cost: ${milk_coffee.get_cost()}\n")

    # Coffee + Milk + Sugar
    sugar_milk_coffee = SugarDecorator(MilkDecorator(PlainCoffee()))
    print(f"Description: {sugar_milk_coffee.get_description()}")
    print(f"Cost: ${sugar_milk_coffee.get_cost()}\n")

    # Coffee + Sugar + Milk + Whipped Cream (order bhi change kar sakte ho!)
    fancy_coffee = WhippedCreamDecorator(SugarDecorator(MilkDecorator(PlainCoffee())))
    print(f"Description: {fancy_coffee.get_description()}")
    print(f"Cost: ${fancy_coffee.get_cost()}")


main()

"""
EXPLANATION — Ye BEST APPROACH kyun better hai bad approach se:

✅ NO CLASS EXPLOSION: Sirf 4 classes (Plain, Milk, Sugar, Cream) banayi,
   lekin unse 2^4 = 16 se bhi zyada combinations bana sakte hain, kisi
   nayi class ki zarurat nahi.

✅ RUNTIME FLEXIBILITY: Jo bhi combination customer chahe, wo turant
   object banate time decide ho jata hai — koi predefined class ki zarurat nahi.

✅ ANY ORDER POSSIBLE: MilkDecorator(SugarDecorator(...)) ya
   SugarDecorator(MilkDecorator(...)) — dono valid hain, order flexible hai.

✅ OPEN/CLOSED PRINCIPLE: Naya add-on (jaise ChocolateDecorator) add karna ho,
   to sirf EK NAYI class banao — PURANA CODE (PlainCoffee, MilkDecorator, etc.)
   BILKUL TOUCH NAHI KARNA PADTA.

Yahi Decorator Pattern ki asli power hai — "wrap karo, modify mat karo"!
"""


## 🌍 Real-World Applications

| Application | Kaise use hota hai |
|---|---|
| ☕ **Coffee Shop Apps** | Milk, Sugar, Whipped Cream jaise add-ons dynamically add hote hain |
| 🎥 **Video Streaming Platforms** | Subtitles, audio enhancements, language options video object ko wrap karte hain |
| ✍️ **Text Processing Apps** | Bold, Italic, Underline formatting text pe wrap hoti hai (combine bhi ho sakti hai) |
| 📂 **Java I/O Streams** | `FileInputStream` ko `BufferedInputStream`/`DataInputStream` se wrap karke naye features milte hain |


## ✅ Advantages (Fayde)

| # | Advantage | Explanation |
|---|---|---|
| 1 | Runtime flexibility | Behavior add/remove kar sakte ho bina class change kiye |
| 2 | Single Responsibility Principle | Har decorator sirf ek feature ka zimmedar hota hai |
| 3 | No class explosion | Combinations ke liye alag-alag subclasses nahi banani padti |
| 4 | Open/Closed Principle | Naya add-on = nayi class, purana code untouched |

## ❌ Disadvantages (Nuksan)

| # | Disadvantage | Explanation |
|---|---|---|
| 1 | Zyada small classes | Har feature ke liye ek chhoti decorator class banani padti hai |
| 2 | Debugging mushkil | Multiple layers wrap hone se flow samajhna thoda tricky ho sakta hai |
| 3 | Overuse risk | Zyada decorators stack karne se code samajhna mushkil ho sakta hai |


---
# 📝 Practice Questions (5 Real-World Examples)

Har question mein: **Problem Statement → Try Yourself → Solution Code → Explanation**

Beginners: pehle khud try karo, phir solution dekho! 💪


## Q1. 🍕 Pizza Topping Decorator

**Problem:** Ek Pizza shop app banao jisme base Pizza pe **Cheese**, **Mushroom**, aur 
**Olives** jaise toppings Decorator Pattern se add ho sake, jaise document mein Capsicum 
aur Cheese Burst diagram mein dikhaya gaya tha.

**Try yourself first**, phir neeche solution dekho.


In [ ]:
"""
SOLUTION — Pizza Topping Decorator
========================================
"""

from abc import ABC, abstractmethod

# Component
class Pizza(ABC):
    @abstractmethod
    def get_description(self):
        pass
    @abstractmethod
    def get_cost(self):
        pass

# Concrete Component
class PlainPizza(Pizza):
    def get_description(self):
        return "Plain Pizza"
    def get_cost(self):
        return 150.0

# Abstract Decorator
class ToppingDecorator(Pizza):
    def __init__(self, wrapped_pizza):
        self.wrapped_pizza = wrapped_pizza
    def get_description(self):
        return self.wrapped_pizza.get_description()
    def get_cost(self):
        return self.wrapped_pizza.get_cost()

# Concrete Decorators
class CheeseTopping(ToppingDecorator):
    def get_description(self):
        return self.wrapped_pizza.get_description() + ", Cheese"
    def get_cost(self):
        return self.wrapped_pizza.get_cost() + 40.0

class MushroomTopping(ToppingDecorator):
    def get_description(self):
        return self.wrapped_pizza.get_description() + ", Mushroom"
    def get_cost(self):
        return self.wrapped_pizza.get_cost() + 30.0

class OlivesTopping(ToppingDecorator):
    def get_description(self):
        return self.wrapped_pizza.get_description() + ", Olives"
    def get_cost(self):
        return self.wrapped_pizza.get_cost() + 25.0


# Client
my_pizza = OlivesTopping(MushroomTopping(CheeseTopping(PlainPizza())))
print(f"Order: {my_pizza.get_description()}")
print(f"Total Cost: Rs.{my_pizza.get_cost()}")

"""
EXPLANATION:
Bilkul document ke Pizza + Capsicum + Cheese Burst example jaisa hi hai —
har topping ek layer add karti hai (description mein naam, cost mein price),
aur andar wali pizza ko touch tak nahi karti. Customer kisi bhi combination
mein topping choose kar sakta hai.
"""


## Q2. ✍️ Text Formatter Decorator

**Problem:** Ek Text Editor feature banao jisme plain text ko **Bold**, **Italic**, aur 
**Underline** formatting Decorator Pattern se apply ki ja sake — jitni chaho utni combine karke.


In [ ]:
"""
SOLUTION — Text Formatter Decorator
==========================================
"""

from abc import ABC, abstractmethod

# Component
class Text(ABC):
    @abstractmethod
    def render(self):
        pass

# Concrete Component
class PlainText(Text):
    def __init__(self, content):
        self.content = content
    def render(self):
        return self.content

# Abstract Decorator
class TextDecorator(Text):
    def __init__(self, wrapped_text):
        self.wrapped_text = wrapped_text
    def render(self):
        return self.wrapped_text.render()

# Concrete Decorators
class BoldDecorator(TextDecorator):
    def render(self):
        return f"<b>{self.wrapped_text.render()}</b>"

class ItalicDecorator(TextDecorator):
    def render(self):
        return f"<i>{self.wrapped_text.render()}</i>"

class UnderlineDecorator(TextDecorator):
    def render(self):
        return f"<u>{self.wrapped_text.render()}</u>"


# Client
simple = PlainText("Hello World")
print("Plain:", simple.render())

bold_text = BoldDecorator(simple)
print("Bold:", bold_text.render())

bold_italic_underline = UnderlineDecorator(ItalicDecorator(BoldDecorator(simple)))
print("All combined:", bold_italic_underline.render())

"""
EXPLANATION:
Har decorator apni HTML tag layer add kar raha hai, andar wale content ko
bina modify kiye. Bold -> Italic -> Underline order mein wrap karke humein
"<u><i><b>Hello World</b></i></u>" mila. Order badal ke result bhi badal
sakta hai (jaise real text editors mein hota hai).
"""


## Q3. 🔔 Multi-Channel Notification Decorator

**Problem:** Ek notification system hai jisme base notification hoti hai, aur usko 
**SMS** ya **Slack** channel se bhi bhejna ho to Decorator se wrap karke **ek hi call mein 
multiple channels** pe bhej sake.


In [ ]:
"""
SOLUTION — Multi-Channel Notification Decorator
======================================================
"""

from abc import ABC, abstractmethod

# Component
class Notifier(ABC):
    @abstractmethod
    def send(self, message):
        pass

# Concrete Component
class BasicNotifier(Notifier):
    def send(self, message):
        print(f"📧 Email sent: {message}")

# Abstract Decorator
class NotifierDecorator(Notifier):
    def __init__(self, wrapped_notifier):
        self.wrapped_notifier = wrapped_notifier
    def send(self, message):
        self.wrapped_notifier.send(message)

# Concrete Decorators
class SMSDecorator(NotifierDecorator):
    def send(self, message):
        self.wrapped_notifier.send(message)          # pehle andar wale ko bhejo
        print(f"📱 SMS sent: {message}")               # phir apna extra kaam karo

class SlackDecorator(NotifierDecorator):
    def send(self, message):
        self.wrapped_notifier.send(message)
        print(f"💬 Slack message sent: {message}")


# Client — Email + SMS + Slack, sab ek hi call mein!
notifier = SlackDecorator(SMSDecorator(BasicNotifier()))
notifier.send("Server is down! Please check immediately.")

"""
EXPLANATION:
Yaha decorators sirf VALUE modify nahi kar rahe (jaise cost/description),
balki EXTRA ACTION bhi perform kar rahe hain (SMS/Slack bhejna) — ye dikhata
hai ki Decorator sirf "data wrap karna" nahi, "extra behavior add karna" bhi
hai. Ek hi .send() call se teeno channels pe message chala gaya!
"""


## Q4. 🤔 Conceptual Question (No Code)

**Question:** Ek beginner developer bolta hai — *"Maine to simple inheritance use karke bhi 
CoffeeWithMilkAndSugar class bana li thi (jaisa humne Bad Approach mein dekha), wo bhi to 
kaam kar raha tha na? Decorator Pattern ki zarurat kya hai?"*

### ✅ Answer:

| Aspect | Inheritance (Bad Approach) | Decorator Pattern (Best Approach) |
|---|---|---|
| **Combinations** | Har combination ke liye NAYI class chahiye (class explosion) | Same chhoti classes se anek combinations bana sakte ho |
| **Flexibility** | Compile-time pe FIXED — object banne ke baad behavior change nahi ho sakta | Runtime pe flexible — jab chaho decorator laga/hata sakte ho |
| **Order Control** | Order class hierarchy mein FIXED hota hai | Wrapping order client decide karta hai (jo chaho pehle wrap karo) |
| **Code Reuse** | Har subclass mein similar logic REPEAT hota hai | Ek decorator ka logic baar baar reuse hota hai, kisi bhi component pe |
| **Open/Closed Principle** | ❌ Naya combo = naya class likhna/modify karna | ✅ Naya feature = ek nayi decorator class, purana code safe |

**Simple yaad rakhne ka tarika:** Inheritance = "ek fixed dress jo tailor pehle se sila hua hai
(badalna mushkil)". Decorator = "ek jacket jo tum kisi bhi outfit ke upar pehan sakte ho,
jab chaho utaar sakte ho, kisi bhi combination mein!"

**Extra Insight:** Inheritance "IS-A" relationship banata hai (CoffeeWithMilk IS-A Coffee),
jabki Decorator "HAS-A" relationship use karta hai (Decorator HAS-A Coffee reference) —
isi wajah se ye zyada flexible hota hai (composition over inheritance principle).


## Q5. 🧪 Debug Karo: Ye Code Kyun Problematic Hai?

**Problem:** Neeche diya gaya code ek **naya developer** ne likha hai jo video streaming 
platform ke liye Subtitles aur HD-Quality features add karna chahta hai, lekin usne 
**Decorator Pattern na use karke** inheritance se try kiya. Pehle problem identify karo, 
phir Decorator Pattern se fix karo.


In [ ]:
"""
❌ PROBLEMATIC CODE (Try to spot the issues before scrolling to solution!)
================================================================================

class Video:
    def play(self):
        return "Playing video"

class VideoWithSubtitles(Video):
    def play(self):
        return super().play() + " with Subtitles"

class VideoWithSubtitlesAndHD(VideoWithSubtitles):
    def play(self):
        return super().play() + " in HD"

# Problem: Agar user sirf "HD without subtitles" chahta hai, to koi class nahi hai!
# Naya combination chahiye -> naya class likhna padega.
"""

# -----------------------------------------------------------------
# ✅ BEST APPROACH — Decorator Pattern se FIX kiya gaya
# -----------------------------------------------------------------

from abc import ABC, abstractmethod

# Component
class Video(ABC):
    @abstractmethod
    def play(self):
        pass

# Concrete Component
class BasicVideo(Video):
    def play(self):
        return "Playing video"

# Abstract Decorator
class VideoDecorator(Video):
    def __init__(self, wrapped_video):
        self.wrapped_video = wrapped_video
    def play(self):
        return self.wrapped_video.play()

# Concrete Decorators
class SubtitlesDecorator(VideoDecorator):
    def play(self):
        return self.wrapped_video.play() + " with Subtitles"

class HDDecorator(VideoDecorator):
    def play(self):
        return self.wrapped_video.play() + " in HD"


# Client — ab HAR combination possible hai, koi nayi class ki zarurat nahi!
print("Only HD:", HDDecorator(BasicVideo()).play())
print("Only Subtitles:", SubtitlesDecorator(BasicVideo()).play())
print("Both:", HDDecorator(SubtitlesDecorator(BasicVideo())).play())

"""
EXPLANATION — Bad code mein PROBLEM kya thi:
1. "HD without Subtitles" jaisa combination missing tha — kyunki
   VideoWithSubtitlesAndHD hamesha PEHLE Subtitles, PHIR HD chain follow karti hai.
2. Naya feature (jaise "AudioBoost") add karna ho to EXISTING classes ko
   dobara likhna padta — risky aur error-prone.
3. Class hierarchy rigid thi — sirf PREDEFINED combinations hi possible the.

BEST APPROACH mein FIX kaise hua:
- Har feature (Subtitles, HD) ab ek INDEPENDENT decorator hai
- Kisi bhi ORDER mein, kisi bhi COMBINATION mein wrap kar sakte ho
- Naya feature add karna ho to sirf EK NAYI decorator class chahiye,
  BasicVideo ya existing decorators ko HAATH TAK NAHI LAGANA PADTA ✅
"""


---
## 🎯 Quick Summary (Revision ke liye)

- **Decorator Pattern** object mein **runtime pe naya behavior** add karta hai, without modifying original class
- 4 components: **Component (Interface), Concrete Component, Decorator (Abstract), Concrete Decorator**
- **Bad approach (inheritance)** mein class explosion hota hai aur combinations FIXED ho jaate hain
- **Best approach (Decorator)** mein jitne chaho utne features, kisi bhi order mein, RUNTIME pe combine kar sakte ho
- Decorator "**composition over inheritance**" principle follow karta hai — HAS-A > IS-A

### Decorator vs Builder vs Factory Method (Quick Recall)
| Pattern | Type | Focus |
|---|---|---|
| **Singleton** | Creational | Sirf EK instance guarantee karna |
| **Builder** | Creational | Complex object ko STEP-BY-STEP banana |
| **Factory Method** | Creational | Object creation SUBCLASS ko delegate karna |
| **Decorator** | Structural | Object mein RUNTIME pe naya BEHAVIOR add karna (wrapping) |

### Real projects mein kaha milega?
- Coffee Shop / Food ordering apps (add-ons)
- Video Streaming platforms (subtitles, audio, quality)
- Text Editors (bold, italic, underline)
- Java I/O Streams (`BufferedInputStream`, `DataInputStream`)
- Python's own `@decorator` syntax (functions ko wrap karna — similar concept!)

Happy Learning! 🚀
